Hier feature engineering un einen gemeinsamen train Datensatz für die Modellierung
- Zielvariable: sales (nur im Train‑Set)

- Extrahiere einen Val Datensatz (die letzten 2 Wochen, z.B.)

- Features:

--Zeitfeatures (Tag, Monat, Wochentag, is_weekend, etc.)

--Store‑Features (Typ, State, Cluster … aus stores)

--Feiertags‑Features (aus holidays_events – zunächst einfache Flags)

--Ölpreis (dcoilwtico ohne Missing Values)

--Transaktionen (transactions)


In [81]:
from pathlib import Path
import pandas as pd
import numpy as np

In [82]:
BASE_DIR = Path(r"C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

files_processed = {
    "train": "train_processed.csv",
    "test": "test_processed.csv",
    "stores": "stores_processed.csv",
    "oil": "oil_processed.csv",
    "holidays_events": "holidays_events_processed.csv",
    "transactions": "transactions_processed.csv",
}

In [83]:
dfs = {}
for name, fname in files_processed.items():
    path = PROCESSED_DIR / fname
    df = pd.read_csv(path, parse_dates=["date"]) if "date" in pd.read_csv(path, nrows=1).columns else pd.read_csv(path)
    dfs[name] = df
    print(f"{name}: shape={df.shape}, date dtype={dfs[name]['date'].dtype if 'date' in df.columns else 'no date column'}")
    print(f"{name}: summary={df.info()} ")

train: shape=(3000888, 6), date dtype=datetime64[ns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[ns]
 2   store_nbr    int64         
 3   family       object        
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 137.4+ MB
train: summary=None 
test: shape=(28512, 5), date dtype=datetime64[ns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           28512 non-null  int64         
 1   date         28512 non-null  datetime64[ns]
 2   store_nbr    28512 non-null  int64         
 3   family       28512 non-null  object        
 4   onpromotio

In [84]:
dfs["holidays_events"].head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [ ]:
## Erstelle Zeit-Features auf train /test Datensatz. Feiertage später

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek  # 0=Montag
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["dayofyear"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    return df

train_fe = add_time_features(train)
test_fe = add_time_features(test)


In [ ]:
train_fe.info()

In [ ]:
train_fe.head(5)

In [ ]:
## Mergen der Öl-Daten

train_fe = train_fe.merge(stores, on="store_nbr", how="left")
test_fe = test_fe.merge(stores, on="store_nbr", how="left")


In [ ]:
train_fe.head(5)

In [ ]:
train_fe.info()

In [ ]:
train_fe.head(5)

In [ ]:
## Transaktionen mergen
## Transaktionsdaten sind auf Ebene (date, store_nbr) und geben zusätzliche Information zur Nachfrage.


train_fe = train_fe.merge(transactions, on=["date", "store_nbr"], how="left")
test_fe = test_fe.merge(transactions, on=["date", "store_nbr"], how="left")

train_fe["transactions"] = train_fe["transactions"].fillna(0)
test_fe["transactions"] = test_fe["transactions"].fillna(0)


In [ ]:
 train_fe["transactions"].describe()

In [ ]:
## Grundlage des Modellierungs-Datensatzes speichern

train_fe.to_csv(PROCESSED_DIR / "train_model_base.csv", index=False)
test_fe.to_csv(PROCESSED_DIR / "test_model_base.csv", index=False)


In [ ]:
## Weitere Zeit-Features: Lags und Rolling-Means pro store/family

## Wichtig: Sortierung nach store, family und date. Sonst stimmen die Lags nicht

train_fe = train_fe.sort_values(["store_nbr", "family", "date"])

In [ ]:
import numpy as np

def add_lag_features(
    df: pd.DataFrame,
    group_cols: list[str],
    target_col: str = "sales",
    lags: list[int] = [1, 7, 14],
    rolling_windows: list[int] = [7, 14, 28],
) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(group_cols + ["date"])

    # Lag-Features
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = (
            df.groupby(group_cols)[target_col].shift(lag)
        )

    # Rolling-Mean-Features
    for lag in lags:
        lag_col = f"{target_col}_lag_{lag}"
        for win in rolling_windows:
            df[f"{target_col}_rmean_lag{lag}_win{win}"] = (
                df.groupby(group_cols)[lag_col]
                  .transform(lambda s: s.rolling(window=win, min_periods=1).mean())
            )

    return df

train_all_lagged = add_lag_features(
    train_fe,
    group_cols=["store_nbr", "family"],
    target_col="sales",
    lags=[1, 7, 14],
    rolling_windows=[7, 28],
)


In [ ]:
train_all_lagged.head(5)

In [71]:
val_part.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81972 entries, 2918916 to 3000755
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       81972 non-null  int64         
 1   date                     81972 non-null  datetime64[ns]
 2   store_nbr                81972 non-null  int64         
 3   family                   81972 non-null  object        
 4   sales                    81972 non-null  float64       
 5   onpromotion              81972 non-null  int64         
 6   year                     81972 non-null  int32         
 7   month                    81972 non-null  int32         
 8   day                      81972 non-null  int32         
 9   dayofweek                81972 non-null  int32         
 10  weekofyear               81972 non-null  int64         
 11  dayofyear                81972 non-null  int32         
 12  is_weekend               8197

In [72]:
## Hier Behandlung der Holiday-Angaben. Die Competition‑Beschreibung sagt explizit, 
## dass der tatsächlich gefeierte Tag auf der Transfer‑Zeile liegt und der ursprüngliche Tag wie ein normaler Tag behandelt werden soll.

# ursprüngliche (verschobene) Holidays
original = dfs["holidays_events"][(holidays_events["transferred"] == True) & (holidays_events["type"] != "Transfer")][
    ["description", "date"]
].rename(columns={"date": "original_date"})

# Zeilen mit type == "Transfer" (tatsächlicher Feiertag)
transfer = holidays_events[holidays_events["type"] == "Transfer"][["description", "date"]].rename(
    columns={"date": "observed_date"}
)

transfer_map = pd.merge(transfer, original, on="description", how="left")
observed_dates = transfer_map["observed_date"].dropna().unique()

NameError: name 'holidays_events' is not defined

In [ ]:
#  Flags pro Holiday-Typ

hol_feat = holidays_events.copy()

hol_feat["is_additional_holiday"] = (hol_feat["type"] == "Additional").astype(int)
hol_feat["is_bridge_day"] = (hol_feat["type"] == "Bridge").astype(int)
hol_feat["is_work_day_comp"] = (hol_feat["type"] == "Work Day").astype(int)
hol_feat["is_transfer_row"] = (hol_feat["type"] == "Transfer").astype(int)
hol_feat["is_transferred_flag"] = hol_feat["transferred"].astype(int)


In [ ]:
# „Observed Holiday“ pro Datum

hol_feat["is_holiday_observed"] = 0

# 1) Tage mit Transfer-Typ (observed_date)
hol_feat.loc[hol_feat["date"].isin(observed_dates), "is_holiday_observed"] = 1

# 2) „normale“ Feiertage ohne Transfer
non_transfer_holidays = hol_feat[
    (hol_feat["type"].isin(["Holiday", "Additional", "Bridge"])) &
    (hol_feat["transferred"] == False)
]["date"].unique()

hol_feat.loc[hol_feat["date"].isin(non_transfer_holidays), "is_holiday_observed"] = 1


In [ ]:
# Auf Datumsebene aggregieren

agg_funcs = {
    "is_holiday_observed": "max",
    "is_additional_holiday": "max",
    "is_bridge_day": "max",
    "is_work_day_comp": "max",
    "is_transferred_flag": "max",
}

holiday_daily = (
    hol_feat.groupby("date", as_index=False)
            .agg(agg_funcs)
            .sort_values("date")
)

# optional: Namen für spätere, komplexere Features
names_per_day = (
    hol.groupby("date")["description"]
       .apply(lambda x: ";".join(sorted(set(x.astype(str)))))
       .reset_index(name="holiday_names")
)

holiday_daily = holiday_daily.merge(names_per_day, on="date", how="left")
holiday_daily.to_csv(PROCESSED_DIR / "holiday_features_daily.csv", index=False)



In [ ]:
# Holiday-Features in Train/Val einbauen

holiday_daily = pd.read_csv(PROCESSED_DIR / "holiday_features_daily.csv", parse_dates=["date"])

for df_name in ["train_all_lagged"]:
    df = locals()[df_name]
    df = df.merge(holiday_daily, on="date", how="left")
    df[["is_holiday_observed",
        "is_additional_holiday",
        "is_bridge_day",
        "is_work_day_comp",
        "is_transferred_flag"]] = (
        df[["is_holiday_observed",
            "is_additional_holiday",
            "is_bridge_day",
            "is_work_day_comp",
            "is_transferred_flag"]]
        .fillna(0)
    )
    locals()[df_name] = df


In [ ]:
## Abtrennen eines Val-Datensatzes mit 6 Wochen am Ende des Test-Datensatzes:

val_start = pd.Timestamp("2017-07-01")
val_end   = pd.Timestamp("2017-08-31")



In [70]:
#  Behandlung der NaNs in den Lag-Fatures - Ersetzung mit dem Median

lag_cols = [c for c in train_all_lagged.columns if "sales_lag_" in c]

# Median je Lag-Spalte berechnen (nur im Trainingszeitraum, nicht über Val/Test!)
lag_medians = train_all_lagged[lag_cols].median()

# In Train- und Val-Teil anwenden
train_part = train_all_lagged[train_all_lagged["date"] < val_start].copy()
val_part   = train_all_lagged[
    (train_all_lagged["date"] >= val_start) & (train_all_lagged["date"] <= val_end)
].copy()

train_part[lag_cols] = train_part[lag_cols].fillna(lag_medians)
val_part[lag_cols]   = val_part[lag_cols].fillna(lag_medians)


In [ ]:
train_part.info()

In [69]:
train_part["date"].describe()

count                          2918916
mean     2015-04-01 07:00:13.186812672
min                2013-01-01 00:00:00
25%                2014-02-15 00:00:00
50%                2015-04-01 12:00:00
75%                2016-05-16 00:00:00
max                2017-06-30 00:00:00
Name: date, dtype: object